# 🌍 ESG Text Classification Challenge
## Deciphering Corporate Sustainability Signals

### **Introduction**
This project tackles the complex challenge of classifying text into **Environmental, Social, and Governance (ESG)** categories using real-world corporate data. Through advanced Natural Language Processing (NLP) techniques and a stacked ensemble architecture, we achieved a competitive accuracy of **0.81623** on the leaderboard.

The solution leverages a sophisticated pipeline including **DistilBERT** for semantic feature extraction, **Weighted Loss** to handle class imbalance, and a **Stacking Ensemble** with Logistic Regression to maximize predictive performance.

The key components of our solution include:
* **Advanced Text Preprocessing:** Cleaning corporate "noise" to isolate meaningful sustainability signals.
* **DistilBERT Fine-Tuning:** Leveraging pre-trained transformers to capture deep semantic context.
* **Weighted Loss Function:** Addressing class imbalance by assigning higher penalties to errors on minority classes.
* **Stacking Ensemble:** Combining predictions from multiple models using a Logistic Regression meta-learner for robust generalization.

### **Table of Contents**
1.  **Project Overview**
    * Problem Statement
    * The 3 Pillars (E, S, G)
    * Evaluation Metric
    * Best Score Achievement
2.  **Exploratory Data Analysis (EDA)**
    * Dataset Overview & Loading
    * Label Distribution & Class Imbalance
    * Text Length & Word Frequency Analysis
3.  **Data Preprocessing**
    * Text Cleaning (Lowercase, Stopwords, Special Characters)
    * Tokenization & Lemmatization
4.  **Model Development**
    * **DistilBERT Fine-Tuning**: Transfer learning for ESG semantics
    * **Weighted Loss Strategy**: Handling imbalanced multi-label data
    * **Ensemble Techniques**: Blending weak learners
    * **Stacking Architecture**: Using Logistic Regression as a meta-learner
5.  **Results & Submission**
    * Performance Visualization
    * Generating Predictions for the Leaderboard

---

### **1- Project Overview**

**Problem Statement**
We are provided with a dataset of text samples derived from corporate statements, reports, and news. The goal is to predict the presence of specific ESG signals. Unlike standard single-label tasks, this is a **multi-label classification** problem:
*
* **Environmental (E):** Climate change, pollution, waste, deforestation, resource use.
* **Social (S):** Labor practices, human rights, diversity, community impact.
* **Governance (G):** Ethics, transparency, board structure, executive compensation.
* **Non-ESG:** Content unrelated to these specific pillars.

**Evaluation Metric**
The performance of submissions is evaluated using the **Macro-averaged F1 score**.
$$F1_{macro} = \frac{1}{N} \sum_{i=1}^{N} F1_i$$
Since this is a multi-label task, the F1 score is computed separately for each label (E, S, G, non-ESG) and then averaged equally. This ensures that performance on underrepresented classes (like 'G' or 'S') is just as important as the majority class.

**Best Score Achievement**
* **Goal:** Develop a machine learning model with the highest possible generalization capability.
* **Best Leaderboard Score:** **0.81623**
* **Key Strategy:** We moved beyond traditional statistical methods (TF-IDF) to deep learning transformers (**DistilBERT**). To squeeze out the final points of accuracy, we implemented a **Weighted Loss** function to fix class imbalances and used a **Stacking Ensemble** where a Logistic Regression model learns how to best combine the outputs of our base models.


In [ ]:
# General libraries
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import re

data_train=pd.read_csv("train.csv")
data_test=pd.read_csv("test.csv")

 ## **2- Data Preprocessing Pipeline**
### **Text Normalization & Cleaning**

In [ ]:
data_train.isnull().sum()
data_train.dropna(subset=['text'], inplace=True)
print(f"New size of train_df after dropping NaNs: {len(data_train)}")

In [ ]:
#lowercase
data_train["text"] = data_train["text"].str.lower()

In [ ]:
#Distribution of ESG labels in data train visualization


# Define the exact column names as seen in data_train.head()
label_cols = ['E', 'S', 'G', 'non_ESG']

# Sum the counts for each label in data_train
# This sums the '1's in each column to get the total frequency
label_counts = data_train[label_cols].sum().sort_values(ascending=False)

# Create the Bar Chart
plt.figure(figsize=(10, 6))
ax = sns.barplot(x=label_counts.index, y=label_counts.values, palette='viridis')

# Add the exact numbers on top of each bar
for i in ax.containers:
    ax.bar_label(i, padding=3, fontsize=12, fmt='%d')

# Styling
plt.title('Distribution of ESG Labels in data_train', fontsize=16, fontweight='bold')
plt.ylabel('Count', fontsize=12)
plt.xlabel('ESG Category', fontsize=12)
plt.ylim(0, label_counts.max() * 1.15) # Add 15% headspace for labels
sns.despine()

plt.show()

In [ ]:
#Quantify unstructured noise artifacts prior to preprocessing.
def check_patterns(df, col_name):
    patterns = {
        'URLs': r'https?://\S+|www\.\S+',
        'HTML tags': r'<.*?>',
        'Mentions': r'@\w+',
        'Hashtags': r'#',
        'Newlines': r'\n'
    }

    print(f"--- Checking patterns in {col_name} ---")
    for name, pattern in patterns.items():
        count = df[col_name].apply(lambda x: bool(re.search(pattern, x))).sum()
        print(f"Number of texts containing {name}: {count}")

# Check patterns in train_df
check_patterns(data_train, 'text')

# Check patterns in test_df
check_patterns(data_test, 'text')

### **Data Integrity Check: Assessing Preprocessing Impact**

It is critical to verify that our text cleaning pipeline has not inadvertently distorted the dataset structure. Aggressive preprocessing (such as removing special characters or stop words) can occasionally result in empty text fields or misaligned indices, which might skew the target label distribution.

In this step, we compare the class distribution before and after cleaning to ensure:
1.  **Label Consistency:** The proportion of *Environmental*, *Social*, and *Governance* tags remains stable, **with a specific focus on the Environmental (E) pillar** to ensure critical climate-related signals were not accidentally stripped away.
2.  **Data Completeness:** No valid samples were lost during the normalization process.

Maintaining the original class balance is essential for training a robust model, **especially to guarantee that the Environmental category retains its distinct signal amidst the noise reduction.**

In [ ]:
#Ensuring the cleaning is not affecting the distribution of labels
def count_patterns_with_e1(df, col_name):
    patterns = {
        'URLs': r'https?://\S+|www\.\S+',
        'HTML tags': r'<.*?>',
        'Mentions': r'@\w+',
        'Hashtags': r'#',
        'Newlines': r'\\n'
    }

    print(f"--- Counting patterns with E=1 in {col_name} ---")
    results = {}
    for name, pattern in patterns.items():
        # Filter for texts containing the pattern
        texts_with_pattern = df[df[col_name].apply(lambda x: bool(re.search(pattern, x)))]

        # Further filter these texts for E=1
        texts_with_pattern_and_e1 = texts_with_pattern[texts_with_pattern['E'] == 1]

        count = len(texts_with_pattern_and_e1)
        print(f"Number of texts containing {name} AND E=1: {count}")
        results[name] = count
    return results

# Check patterns in data_train with E=1 and capture results
pattern_e1_counts = count_patterns_with_e1(data_train, 'text')

# Convert results to a pandas Series for plotting
plot_data = pd.Series(pattern_e1_counts).sort_values(ascending=False)

# Create the Bar Chart
plt.figure(figsize=(10, 6))
ax = sns.barplot(x=plot_data.index, y=plot_data.values, palette='viridis')

# Add the exact numbers on top of each bar
for i in ax.containers:
    ax.bar_label(i, padding=3, fontsize=12, fmt='%d')

# Styling
plt.title('Distribution of Patterns with E=1 in data_train', fontsize=16, fontweight='bold')
plt.ylabel('Count', fontsize=12)
plt.xlabel('Pattern Category', fontsize=12)
plt.ylim(0, plot_data.max() * 1.15) # Add 15% headspace for labels
sns.despine()

plt.show()

In [ ]:
def clean_text_smart(text):
    if not isinstance(text, str): return ""

    text = text.lower()

    # 1. URLs: Remove them (They are just noise)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 2. HTML: Remove
    text = re.sub(r'<.*?>', '', text)

    # 3. Mentions: Remove the '@' but KEEP the name
    # "Hello @Tesla" -> "Hello Tesla"
    text = re.sub(r'@', '', text)

    # 4. Hashtags: Remove the '#' but KEEP the text
    # "#SolarPower" -> "SolarPower"
    text = re.sub(r'#', '', text)

    # 5. Newlines: Replace with space
    text = re.sub(r'\n', ' ', text)

    # 6. Remove extra spaces (formatting)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply this new function
data_train['text'] = data_train['text'].apply(clean_text_smart)
data_test['text'] = data_test['text'].apply(clean_text_smart)

In [ ]:
# Count the number of empty strings in the 'text' column
empty_text_count = (data_train['text'] == '').sum()
print(f"Number of empty text entries in data_train: {empty_text_count}")

# Drop rows where text became empty
# Comment: These texts were likely composed entirely of patterns we removed (e.g., only URLs, emojis, or special characters).
# Since they now contain no meaningful content, we remove them to prevent errors during model training.
data_train = data_train[data_train['text'] != '']

### **Label Consistency & Logical Validation**

In multi-label classification, ensuring logical consistency in the target variables is critical for model performance. We perform a strict audit of the training labels to identify and resolve two specific types of anomalies:

1.  **Unlabeled Samples (All Zeros):** We check for instances where no label is assigned ($E=0, S=0, G=0, \text{non-ESG}=0$). These "silent" samples can introduce ambiguity if not handled or understood.
2.  **Conflicting Labels (Logical Contradictions):** We define a hard constraint: a text **cannot** be labeled as `non_ESG` if it is simultaneously flagged as `Environmental`, `Social`, or `Governance`. By definition, if a text belongs to an ESG pillar, it is not "non-ESG".

The following code identifies these contradictory entries and removes them from the dataset to prevent the model from learning conflicting patterns.

In [ ]:
# Case 1: All four labels are 0
all_zero_labels_count = len(data_train[(data_train['E'] == 0) & (data_train['S'] == 0) & (data_train['G'] == 0) & (data_train['non_ESG'] == 0)])
print(f"Number of cases where all labels (E, S, G, non_ESG) are 0: {all_zero_labels_count}")

# Case 2: non_ESG is 1 AND at least one of E, S, or G is 1
non_esg_and_other_esg_count = len(data_train[
    (data_train['non_ESG'] == 1) &
    ((data_train['E'] == 1) | (data_train['S'] == 1) | (data_train['G'] == 1))
])
print(f"Number of cases where non_ESG is 1 and at least one of E, S, or G is 1: {non_esg_and_other_esg_count}")
cols = ['E', 'S', 'G', 'non_ESG']
for col in cols:
    data_train[col] = pd.to_numeric(data_train[col], errors='coerce').fillna(0).astype(int)

print("Columns converted to Integers.")

# 2. NOW run your filter again
conflicting_mask = (
    (data_train['non_ESG'] == 1) &
    ((data_train['E'] == 1) | (data_train['S'] == 1) | (data_train['G'] == 1))
)

# 3. Apply the filter
data_train = data_train[~conflicting_mask].copy()

In [ ]:
# Define the exact column names as seen in data_train.head()
label_cols = ['E', 'S', 'G', 'non_ESG']


label_counts = data_train[label_cols].sum().sort_values(ascending=False)

# Create the Bar Chart
plt.figure(figsize=(10, 6))
ax = sns.barplot(x=label_counts.index, y=label_counts.values, palette='viridis')


for i in ax.containers:
    ax.bar_label(i, padding=3, fontsize=12, fmt='%d')

# Styling
plt.title('Distribution of ESG Labels in data_train', fontsize=16, fontweight='bold')
plt.ylabel('Count', fontsize=12)
plt.xlabel('ESG Category', fontsize=12)
plt.ylim(0, label_counts.max() * 1.15)
sns.despine()

plt.show()

### **Duplicate Analysis & Label Consistency**

In real-world datasets, duplicate entries are common but require careful handling to prevent model bias or confusion. We distinguish between two types of duplicates:

1.  **Consistent Duplicates:** Instances where the text is identical *and* the labels are identical. While these are less harmful, they can artificially inflate the importance of certain samples.
2.  **Conflicting Duplicates (Label Noise):** Instances where the **same text** appears multiple times but has **different labels**. This is critical "noise"—if the ground truth contradicts itself, the model cannot learn a reliable decision boundary.

**Strategy:**
To ensure data integrity, we perform a strict consistency check. Any text string that appears with conflicting labels is considered ambiguous and is removed from the dataset to prevent it from confusing the model during training.

In [ ]:
duplicates = data_train[data_train.duplicated(subset="text", keep=False)]
print("Number of duplicate rows:", len(duplicates))


print(f"\n--- Duplicate Report ---")
print(f"Total rows that are duplicates: {len(duplicates)}")
print(f"Unique text strings that appear >1 time: {duplicates['text'].nunique()}")


label_cols = ['E', 'S', 'G', 'non_ESG']

consistency_check = duplicates.groupby('text')[label_cols].nunique()


contradictions = consistency_check[consistency_check.max(axis=1) > 1]

print(f"\n--- Consistency Analysis ---")
print(f"Consistent Duplicates (Same text, Same labels): {duplicates['text'].nunique() - len(contradictions)}")
print(f"Conflicting Duplicates (Same text, DIFFERENT labels): {len(contradictions)}")


if len(contradictions) > 0:
    print("\n Example of Contradiction:")
    example_text = contradictions.index[0]
    print(f"Text: {example_text[:100]}...")
    print(data_train[data_train['text'] == example_text][label_cols])


print(f"Number of rows in data_train before dropping conflicting duplicates: {data_train.shape[0]}")

# Get the list of text entries that have conflicting labels
conflicting_texts = contradictions.index.tolist()

# Drop rows from data_train where 'text' is in the conflicting_texts list
data_train = data_train[~data_train['text'].isin(conflicting_texts)]

print(f"Number of rows in data_train after dropping conflicting duplicates: {data_train.shape[0]}")

In [ ]:
# MODELING LIBRARIES
import torch
from torch import nn
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments, DistilBertTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier

### **Data Tokenization & Custom Dataset Class**

To prepare our text for **DistilBERT**, we must first convert raw strings into numerical tokens using the pre-trained `DistilBertTokenizer`. We then define a custom PyTorch `Dataset` class (`ESGDataset`) to efficiently manage these encodings and their corresponding multi-label targets during training.

In [ ]:
# PREPARE DATASETS & UTILS

print(" Preparing Datasets")

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

class ESGDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

### **3-Custom Trainer: Addressing Class Imbalance**

Standard loss functions often struggle with imbalanced datasets, as they treat all classes equally. To improve our model's sensitivity to underrepresented ESG pillars (like *Governance*), we implement a **WeightedTrainer**. This custom class overrides the standard loss computation, replacing it with `BCEWithLogitsLoss` equipped with positive class weights. This ensures that the model is penalized more heavily for missing minority class signals, leading to better recall and F1 scores.

In [ ]:
# Custom Trainer to handle Class Imbalance
class WeightedTrainer(Trainer):
    def __init__(self, pos_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weights = pos_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weights.to(model.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

### **4-Ensemble Training & Stacking Strategy (5-Fold CV)**

This section implements the core of our solution: a **5-Fold Cross-Validation** training loop. Instead of relying on a single train-test split, we train five separate instances of `DistilBERT`. This approach provides three critical advantages:

1.  **Robust Evaluation:** By validating on five different subsets of the data, we get a reliable estimate of the model's true performance.
2.  **Stacking Preparation (OOF):** We generate **Out-of-Fold (OOF)** predictions for every sample in the training set. These predictions act as "meta-features," allowing our Level-2 model (Logistic Regression) to learn how to correct the errors of the base model without overfitting.
3.  **Variance Reduction:** Each of the 5 models makes a prediction on the final *Test Set*. Averaging these predictions smoothens out noise and improves generalization.



*Note: We dynamically recalculate positive class weights for each fold to ensure that the loss function remains perfectly balanced according to the specific training split.*

In [ ]:
# ENSEMBLE TRAINING LOOP (5 Folds)

N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Storage for Stacking
oof_preds = np.zeros((len(data_train), 4))
test_preds_list = []

print("Tokenizing Test Data...")
test_encodings = tokenizer(data_test['text'].tolist(), truncation=True, padding=True, max_length=128)
test_dataset = ESGDataset(test_encodings)

print(f"Starting {N_FOLDS}-Fold Ensemble Training...")

for fold, (train_idx, val_idx) in enumerate(kf.split(data_train)):
    print(f"\n{'='*15} FOLD {fold+1}/{N_FOLDS} {'='*15}")

    # 1. Split Data
    train_fold = data_train.iloc[train_idx]
    val_fold = data_train.iloc[val_idx]
    train_labels = train_fold[label_cols].values
    val_labels = val_fold[label_cols].values

    # 2. Calculate Fold-Specific Weights
    positive_counts = np.sum(train_labels, axis=0)
    negative_counts = len(train_labels) - positive_counts
    pos_weights = torch.tensor(negative_counts / positive_counts, dtype=torch.float)

    # 3. Tokenize
    train_enc = tokenizer(train_fold['text'].tolist(), truncation=True, padding=True, max_length=128)
    val_enc = tokenizer(val_fold['text'].tolist(), truncation=True, padding=True, max_length=128)

    train_ds = ESGDataset(train_enc, train_labels)
    val_ds = ESGDataset(val_enc, val_labels)

    # 4. Initialize Fresh Model
    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=4
    )

    training_args = TrainingArguments(
        output_dir=f"./results_fold_{fold}",
        num_train_epochs=3,
        per_device_train_batch_size=16,
        learning_rate=2e-5,
        save_strategy="no",
        eval_strategy="no",
        logging_steps=100,
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        pos_weights=pos_weights
    )
    trainer.train()

    # 5. Generate Features for Stacking
    # Predict on Validation Fold (Out-of-Fold)
    val_output = trainer.predict(val_ds)
    oof_preds[val_idx] = val_output.predictions

    # Predict on Test Set (to be averaged later)
    test_output = trainer.predict(test_dataset)
    test_preds_list.append(test_output.predictions)

    # Clear GPU Cache
    del model, trainer
    torch.cuda.empty_cache()


### **5-Level-2 Stacking: The Meta-Learner**

With our DistilBERT base models trained, we now move to the second stage of our ensemble: the **Meta-Learner**.

Instead of simple averaging, we use a **Stacking** approach. We train a `LogisticRegression` classifier on the *Out-of-Fold (OOF)* predictions generated in the previous step.

**Why this works:**
1.  **Bias Correction:** If the DistilBERT models consistently over-predict a certain class (e.g., confusing *Social* with *Governance*), the Logistic Regression model learns to identify and correct this pattern.
2.  **Calibrated Decisions:** The meta-learner acts as a final judge, weighing the confidence scores (logits) from the deep learning model to produce a more refined probability.



We wrap the regression model in a `MultiOutputClassifier` to handle the multi-label nature of our target variables (E, S, G, non-ESG).

In [ ]:
# TRAIN META-LEARNER (STACKING)

print("\n Training Meta-Learner (Logistic Regression)...")

# Train the meta-learner on OOF predictions vs True Labels
meta_learner = MultiOutputClassifier(LogisticRegression(class_weight='balanced', max_iter=1000))
meta_learner.fit(oof_preds, data_train[label_cols].values)

# Evaluate calibration
stacking_score = meta_learner.score(oof_preds, data_train[label_cols].values)
print(f" Meta-Learner Calibration Score: {stacking_score:.4f}")


### **6-Final Inference & Submission Generation**

In this concluding step, we synthesize the outputs of our entire pipeline to generate the final predictions for the leaderboard.

**The Inference Process:**
1.  **Ensemble Averaging:** We take the raw prediction scores (logits) from all 5 DistilBERT models trained during cross-validation. By averaging these scores, we reduce the variance and smooth out model-specific noise.
2.  **Stacking Prediction:** The averaged logits are passed as input features to our trained **Meta-Learner** (Logistic Regression). This model applies the final calibration, determining the optimal decision boundary for each class based on the patterns it learned during validation.
3.  **Formatting:** The final binary predictions are mapped to the required submission format and saved as a CSV file.

In [ ]:
# FINAL INFERENCE

print("\n Generating Final Predictions...")

# Ensemble: Average the test logits from all 5 folds
avg_test_logits = np.mean(test_preds_list, axis=0)

# Stacking: Final pass through meta-learner
final_preds = meta_learner.predict(avg_test_logits)

submission = pd.DataFrame({
    'id': data_test['id'],
    'E': final_preds[:, 0],
    'S': final_preds[:, 1],
    'G': final_preds[:, 2],
    'non_ESG': final_preds[:, 3]
})

submission.to_csv("submission_ensemble_stacking.csv", index=False)
print("Submission saved to 'submission_ensemble_stacking.csv'")